# Предсказание стоимости жилья

<font size="3"> Необходимо построить модель линейной регрессии, которая поможет предсказать медианную стоимость дома в жилом массиве. Для оценки качества модели необходимо использовать метрики RMSE, MAE и R2.

**Для проведения исследования доступны следующие данные:** 
- датасет с данными о жилье в Калифорнии в 1990 году.

**План проведения исследования:**
- изучим данные;
- проведём предобработку;
- построим две модели линейной регрессии на разных наборах данных;
- качество модели оценим метриками метрики RMSE, MAE и R2;
- сформулируем выводы и предложения.

---

## Загрузка и чтение данных

### Подготовка к чтению данных

<font size="3"> Импортируем необходимые библиотеки и инициализируем локальную Spark-сессию.

In [1]:
import numpy as np
import pandas as pd

import pyspark
import pyspark.sql.functions as F
from pyspark.sql import Row
from pyspark.sql import SparkSession
from pyspark.sql.types import *

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

pyspark_version = pyspark.__version__
if int(pyspark_version[:1]) == 3:
    from pyspark.ml.feature import OneHotEncoder
elif int(pyspark_version[:1]) == 2:
    from pyspark.ml.feature import OneHotEncodeEstimator as OneHotEncoder

In [2]:
spark = SparkSession.builder \
                    .master("local") \
                    .appName("EDA California Housing") \
                    .getOrCreate()

### Чтение данных

<font size="3"> Считаем данные о жилье в Калифорнии в 1990 году и сохраним в переменную `df_housing`:

In [3]:
df_housing = spark.read.load('/datasets/housing.csv', format="csv", sep=",", inferSchema=True, header="true")

<font size="3"> Проверим результат:

In [4]:
df_housing.printSchema()
df_housing.show(5)

root
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- median_house_value: double (nullable = true)
 |-- ocean_proximity: string (nullable = true)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|  -122.23|   37.88|              41.0|      880.0|         129.0|     322.0|     126.0|       8.3252|          452600.0|       NEAR B

### Изучение данных

<font size="3"> Выведем типы данных колонок полученного датафрейма:

In [5]:
print(pd.DataFrame(df_housing.dtypes, columns=['column', 'type']))

               column    type
0           longitude  double
1            latitude  double
2  housing_median_age  double
3         total_rooms  double
4      total_bedrooms  double
5          population  double
6          households  double
7       median_income  double
8  median_house_value  double
9     ocean_proximity  string


<font size="3"> Выведем количество пропущенных значений в столбцах датафрейма:

In [6]:
for col in df_housing.columns:
    missing_count = df_housing.filter(F.col(col).isNull()).count()
    print(f"Количество пропущенных значений в столбце '{col}': {missing_count}")

Количество пропущенных значений в столбце 'longitude': 0
Количество пропущенных значений в столбце 'latitude': 0
Количество пропущенных значений в столбце 'housing_median_age': 0
Количество пропущенных значений в столбце 'total_rooms': 0
Количество пропущенных значений в столбце 'total_bedrooms': 207
Количество пропущенных значений в столбце 'population': 0
Количество пропущенных значений в столбце 'households': 0
Количество пропущенных значений в столбце 'median_income': 0
Количество пропущенных значений в столбце 'median_house_value': 0
Количество пропущенных значений в столбце 'ocean_proximity': 0


<font size="3">
    
**Вывод:**

Датафрейм `df_housing` содержит следующие данные:

  * `longitude` — широта;
  * `latitude` — долгота;
  * `housing_median_age` — медианный возраст жителей жилого массива;
  * `total_rooms` — общее количество комнат в домах жилого массива;  
  * `total_bedrooms` — общее количество спален в домах жилого массива;  
  * `population` — количество человек, которые проживают в жилом массиве;  
  * `households` — количество домовладений в жилом массиве;  
  * `median_income` — медианный доход жителей жилого массива;  
  * `median_house_value` — медианная стоимость дома в жилом массиве;  
  * `ocean_proximity` — близость к океану.

Данных достаточно для проведения исследования. 
    
В датасете имеются пропущенные значения, требуется предобработка.

---

## Предобработка данных

<font size="3"> Проведём предобработку данных в датафрейме `df_housing`.

### Проверка пропусков

<font size="3"> В датасете обнаружены пропущенные значения:

In [7]:
for col in df_housing.columns:
    missing_count = df_housing.filter(F.col(col).isNull()).count()
    print(f"Количество пропущенных значений в столбце '{col}': {missing_count}")

Количество пропущенных значений в столбце 'longitude': 0
Количество пропущенных значений в столбце 'latitude': 0
Количество пропущенных значений в столбце 'housing_median_age': 0
Количество пропущенных значений в столбце 'total_rooms': 0
Количество пропущенных значений в столбце 'total_bedrooms': 207
Количество пропущенных значений в столбце 'population': 0
Количество пропущенных значений в столбце 'households': 0
Количество пропущенных значений в столбце 'median_income': 0
Количество пропущенных значений в столбце 'median_house_value': 0
Количество пропущенных значений в столбце 'ocean_proximity': 0


<font size="3"> Пропуски в столбце `total_bedrooms` заполним медианным значением. 

In [8]:
# Вычислим медианное значение столбца total_bedrooms
median_value = df_housing.approxQuantile("total_bedrooms", [0.5], 0.001)[0]

# Заполним пропущенные значения в столбце total_bedrooms медианным значением
df_housing = df_housing.fillna(median_value, subset=["total_bedrooms"])

# Проверим результат заполнения пропусков
for col in df_housing.columns:
    missing_count = df_housing.filter(F.col(col).isNull()).count()
    print(f"Количество пропущенных значений в столбце '{col}': {missing_count}")

Количество пропущенных значений в столбце 'longitude': 0
Количество пропущенных значений в столбце 'latitude': 0
Количество пропущенных значений в столбце 'housing_median_age': 0
Количество пропущенных значений в столбце 'total_rooms': 0
Количество пропущенных значений в столбце 'total_bedrooms': 0
Количество пропущенных значений в столбце 'population': 0
Количество пропущенных значений в столбце 'households': 0
Количество пропущенных значений в столбце 'median_income': 0
Количество пропущенных значений в столбце 'median_house_value': 0
Количество пропущенных значений в столбце 'ocean_proximity': 0


<font size="3"> Пропуски обработаны.

<font size="3">

**Вывод**

Выполнена предобработка данных:

- заполнены пропуски в датафрейме.

---

## Построение моделей


<font size="3"> Построим две модели линейной регрессии на разных наборах данных:
   * используя все столбцы датафрейма;
   * используя только числовые признаки, исключив категориальные.
   
   
Для построения модели используем оценщик `LinearRegression`.

### Формирование выборок

<font size="3"> Cоздадим тренировочную и тестовую выборки с помощью `randomSplit()`.

In [9]:
# Разделим данные на тренировочную и тестовую выборки для обоих наборов данных
train_data, test_data = df_housing.randomSplit([0.8, 0.2], seed=123)

# Проверим результат
print("Количество строк в тренировочной выборке (все признаки):", train_data.count())
print("Количество строк в тестовой выборке (все признаки):", test_data.count())

Количество строк в тренировочной выборке (все признаки): 16442
Количество строк в тестовой выборке (все признаки): 4198


### Обработка категориальных столбцов

<font size="3"> Преобразуем категориальный столбец `ocean_proximity` техникой `One hot encoding`.

In [10]:
# Преобразуем столбец ocean_proximity в числовой формат с помощью StringIndexer
indexer = StringIndexer(inputCol="ocean_proximity", outputCol="ocean_proximity_idx")
indexer = indexer.fit(train_data)
train_data = indexer.transform(train_data)
test_data = indexer.transform(test_data)

# Применим One-Hot Encoding к числовому столбцу ocean_proximity_index
encoder = OneHotEncoder(inputCol="ocean_proximity_idx", outputCol="ocean_proximity_ohe")
encoder = encoder.fit(train_data)
train_data = encoder.transform(train_data)
test_data = encoder.transform(test_data)

# Проверим результат
print(pd.DataFrame(train_data.dtypes, columns=['column', 'type']))
print(pd.DataFrame(test_data.dtypes, columns=['column', 'type']))

                 column    type
0             longitude  double
1              latitude  double
2    housing_median_age  double
3           total_rooms  double
4        total_bedrooms  double
5            population  double
6            households  double
7         median_income  double
8    median_house_value  double
9       ocean_proximity  string
10  ocean_proximity_idx  double
11  ocean_proximity_ohe  vector
                 column    type
0             longitude  double
1              latitude  double
2    housing_median_age  double
3           total_rooms  double
4        total_bedrooms  double
5            population  double
6            households  double
7         median_income  double
8    median_house_value  double
9       ocean_proximity  string
10  ocean_proximity_idx  double
11  ocean_proximity_ohe  vector


### Подготовка входных признаков

<font size="3"> Соберём два набора входных признаков:
    
- `features_all` - в данную выборку войдут все данные;   
- `numerical_features` - в данную выборку войдут только числовые признаки, исключая категориальные.

In [11]:
# Определим числовые столбцы
numerical_cols = ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
# Определим категориальные столбцы
categorical_cols = ['ocean_proximity_ohe']

# Создадим VectorAssembler для всех признаков
assembler_all = VectorAssembler(inputCols=categorical_cols + numerical_cols, outputCol="features_all")
train_data = assembler_all.transform(train_data)
test_data = assembler_all.transform(test_data)

# Создадим VectorAssembler только для числовых признаков
assembler_numerical = VectorAssembler(inputCols=numerical_cols, outputCol="numerical_features")
train_data = assembler_numerical.transform(train_data)
test_data = assembler_numerical.transform(test_data)

### Обучение моделей

<font size="3"> Построим две модели линейной регрессии на созданных наборах данных:

In [12]:
# Создадим модель линейной регрессии для всех признаков
lr_model_all = LinearRegression(featuresCol="features_all", labelCol='median_house_value')
lr_model_all = lr_model_all.fit(train_data)

# Создадим модель линейной регрессии только для числовых признаков
lr_model_numerical = LinearRegression(featuresCol="numerical_features", labelCol='median_house_value')
lr_model_numerical = lr_model_numerical.fit(train_data)

# Получим предсказания на тестовых данных для обоих моделей
predictions_all = lr_model_all.transform(test_data)
predictions_numerical = lr_model_numerical.transform(test_data)

24/05/27 11:13:40 WARN Instrumentation: [45a7b291] regParam is zero, which might cause numerical instability and overfitting.
24/05/27 11:13:41 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
24/05/27 11:13:41 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS
24/05/27 11:13:41 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeSystemLAPACK
24/05/27 11:13:41 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeRefLAPACK
24/05/27 11:13:43 WARN Instrumentation: [ac63fe12] regParam is zero, which might cause numerical instability and overfitting.


---

## Качество модели

<font size="3"> Сравним результаты работы линейной регрессии на двух наборах данных по метрикам `RMSE`, `MAE` и `R2`.

In [13]:
evaluator = RegressionEvaluator(labelCol="median_house_value", predictionCol="prediction")

# Оценим качество модели, обученной на всех признаках
rmse_all = evaluator.evaluate(predictions_all, {evaluator.metricName: "rmse"})
mae_all = evaluator.evaluate(predictions_all, {evaluator.metricName: "mae"})
r2_all = evaluator.evaluate(predictions_all, {evaluator.metricName: "r2"})

# Оценим качество модели, обученной на количественных признаках
rmse_numerical = evaluator.evaluate(predictions_numerical, {evaluator.metricName: "rmse"})
mae_numerical = evaluator.evaluate(predictions_numerical, {evaluator.metricName: "mae"})
r2_numerical = evaluator.evaluate(predictions_numerical, {evaluator.metricName: "r2"})

<font size="3"> Для удобства соберём полученные метрики в сравнительную таблицу:

In [14]:
# Создадим список с метриками
metrics_data = [
    Row(model="All Features", RMSE=rmse_all, MAE=mae_all, R2=r2_all),
    Row(model="Numerical Features", RMSE=rmse_numerical, MAE=mae_numerical, R2=r2_numerical)
]

# Создадим DataFrame из списка метрик
metrics_df = spark.createDataFrame(metrics_data)

metrics_df.show()

spark.stop()

+------------------+-----------------+-----------------+------------------+
|             model|             RMSE|              MAE|                R2|
+------------------+-----------------+-----------------+------------------+
|      All Features|67539.14614041554|49798.48254932792|0.6505505948634358|
|Numerical Features|68335.11170760056|50795.82907568965|0.6422653635132556|
+------------------+-----------------+-----------------+------------------+



<font size="3">

**Вывод**
    
- `RMSE` - наилучшая метрика получена на модели, обученной на всех входных признаках: корень среднеквадратичной ошибки наименьший и составляет 67 539 USD;
- `MAE` - наилучшая метрика получена на модели, обученной на всех входных признаках: отклонение прогноза от истинного значения целевого признака составило 49 798 USD;
- `R2` - наилучшая метрика получена на модели, обученной на всех входных признаках: модель показывает себя хорошо в 65% случаев.
    
Для получения лучшего прогноза стоит обучить модель на всех входных признаках: числовых и категориальных. 

---

## Общий вывод

**Шаг 1. Изучение данных**

Перед проведением исследования данные были изучены:

- импортированы соответствующие библиотеки и инициализирована локальная Spark-сессию;
- считан датасет с данными о жилье в Калифорнии в 1990 году и сохранён в переменную `df_housing`;
- изучена общая информация о полученных данных.

**Шаг 2. Предобработка данных**

Проведена предобработка данных:

- обработаны пропуски в датафрейме;
- категориальные столбцы преобразованы техникой `One hot encoding`.

**Шаг 3. Построение моделей**

Построены две модели линейной регрессии на разных наборах данных:

- собраны два набора входных признаков: все данные и только количественные признаки;
- для каждого набора признаков созданы тренировочная и тестовая выборки; 
- на выборках обучены две модели линейной регрессии с помощью оценщика `LinearRegression`.

**Шаг 4. Качество моделей**

Результаты работы линейной регрессии на двух наборах данных оценены по метрикам RMSE, MAE и R2.

- `RMSE` - наилучшая метрика получена на всех входных признаках: корень среднеквадратичной ошибки наименьший и составляет 67539 USD;
- `MAE` - наилучшая метрика получена на всех входных признаках: отклоенние прогноза от истинного значения целевого признака составило 49798 USD;
- `R2` - наилучшая метрика получена на всех входных признаках: модель показывает себя хорошо в 65% случаев.

**Заключение**

<font size="3"> Для получения лучшего прогноза стоит обучить модель на всех входных признаках: числовых и категориальных. 
    
Для улучшения качества модели стоит дополнительно провести исследовательский и корреляционный анализ данных, отобрать лучшие признаки по полученным выводам и масштабировать количественные признаки перед обучением модели.